# Modelado Predictivo: Clasificación con Random Forest
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI)

**Contexto:** Tomando como base el dataset limpio, libre de fugas de datos y estandarizado (`CTG_clean.csv`), este notebook implementa y entrena un modelo de **Random Forest** (Bosque Aleatorio). El propósito es aprovechar las 21 variables para clasificar de forma robusta el estado de salud del fetal (`NSP`), reduciendo la varianza y mitigando el sobreajuste (*overfitting*) propio de los árboles de decisión individuales mediante la combinación de *Bagging* (muestreo con reemplazo) y la decorrelación por submuestreo de características (*feature subsampling*).

### Objetivos del Modelado

1. **Construcción y Entrenamiento del Ensamble:** Ensamblar un clasificador `RandomForestClassifier` basado en múltiples árboles de decisión CART, configurando hiperparámetros clave como el número de árboles (`n_estimators`), la profundidad máxima (`max_depth`) y el límite de características por división (`max_features`).

2. **Validación Externa e Interna (OOB Score):** Implementar la validación interna mediante la puntuación *Out-Of-Bag* (`oob_score`), aprovechando las muestras que no participaron en el entrenamiento de cada árbol bootstrap para medir el rendimiento de generalización sin sesgos.

3. **Evaluación de Rendimiento Clínico:** Analizar la matriz de confusión resultante y la capacidad predictiva del modelo frente al severo desbalance de clases del dataset (77.85% Normal vs. 8.28% Patológico), priorizando la detección precisa de riesgos de hipoxia o sufrimiento fetal.

#### **Importación de modelos y preparación de datos**

In [36]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd

# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importamos las clases de los modelos
from src.DecisionTreeClassifier import DecisionTreeClassifier
from src.RandomForest import RandomForestClassifier

In [37]:
# Carga del dataset limpio de cardiotocografía fetal
clean_data_path = '../data/processed/CTG_clean.csv'
df = pd.read_csv(clean_data_path)

# Separación de la matriz de características (X) y el vector objetivo (y)
X = df.drop(columns=["NSP"]).values
y = df["NSP"].values

# Visualización de dataset y dimensiones
display(df.head())
df.info()

print(f"Dimensión de Matriz X (Características): {X.shape}")
print(f"Dimensión de Vector y (Objetivo NSP): {y.shape}")
print(f"Clases en y: {np.unique(y)}")

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,Min,Max,Nmax,Nzeros,Mode,Mean,Median,Variance,Tendency,NSP
0,120.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,73.0,0.5,43.0,...,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006380,0.0,0.006380,0.003190,0.0,0.0,17.0,2.1,0.0,...,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003322,0.0,0.008306,0.003322,0.0,0.0,16.0,2.1,0.0,...,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.002561,0.0,0.007682,0.002561,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.006515,0.0,0.008143,0.000000,0.0,0.0,16.0,2.4,0.0,...,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0


<class 'pandas.DataFrame'>
RangeIndex: 2126 entries, 0 to 2125
Data columns (total 22 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   LB        2126 non-null   float64
 1   AC        2126 non-null   float64
 2   FM        2126 non-null   float64
 3   UC        2126 non-null   float64
 4   DL        2126 non-null   float64
 5   DS        2126 non-null   float64
 6   DP        2126 non-null   float64
 7   ASTV      2126 non-null   float64
 8   MSTV      2126 non-null   float64
 9   ALTV      2126 non-null   float64
 10  MLTV      2126 non-null   float64
 11  Width     2126 non-null   float64
 12  Min       2126 non-null   float64
 13  Max       2126 non-null   float64
 14  Nmax      2126 non-null   float64
 15  Nzeros    2126 non-null   float64
 16  Mode      2126 non-null   float64
 17  Mean      2126 non-null   float64
 18  Median    2126 non-null   float64
 19  Variance  2126 non-null   float64
 20  Tendency  2126 non-null   float64
 21  NS

#### **Entrenamiento y Analisis del (`RandomForestClassifier`)**

In [ ]:
# Entrenamiento del modelo
n_trees_config = 20 #Numero de árboles
max_depth_config = 10 #Profundidad máxima de cada rama

forest = RandomForestClassifier(n_trees=n_trees_config, max_depth=max_depth_config, seed=0)
forest.fit(X, y)

# Cálculo de la precisión Out-Of-Bag (validación interna sin sesgos)
oob_acc = forest.oob_score(X, y)

# Cálculo de la precisión global sobre el conjunto de entrenamiento
train_preds = forest.predict(X)
train_acc = (train_preds == y).mean()

#

print("--- Resultados de Rendimiento ---")
print(f"-> Precisión Out-Of-Bag (OOB Score): {oob_acc * 100:.2f}%")
print(f"-> Precisión Global en Entrenamiento:  {train_acc * 100:.2f}%")

--- Resultados de Rendimiento ---
-> Precisión Out-Of-Bag (OOB Score): 93.13%
-> Precisión Global en Entrenamiento:  98.45%
